# Workflows

Sometimes, having to call multiple request and retrieve information to do consecutive requests can be tedious. Workflows allow you to chain multiple requests together, by using predefined steps.

## Creating a Workflow

In [1]:
from silkroute import Workflow

wf = Workflow()

To run a workflow object we need to first define the parameters of the workflow. The main ones are:

- modality: Depending on the type of dataset we want to retrieve, we can select between 'protein', 'compound' or 'interaction'. protein will retrieve protein sequences from UniProt, compound will retrieve small molecule structures from ChEMBL, and interaction will retrieve either protein-protein or protein-ligand interactions.
    
- mode: The mode defines the order of operations. There are three modes available:

    - query_first: In this mode, we first perform a query to retrieve identifiers, and then use those identifiers to retrieve the data.
        
    - import_first: In this mode, we first import a list of identifiers, and then use those identifiers to perform a query.
        
    - query_composition: In this mode, we perform multiple queries with labels, and then combine the results.
    
- query: For 'query_first' and 'query_composition' modes, we need to provide a query string to search for identifiers.
    
- enrich: Whether to perform enrichment of the retrieved data with additional information from other databases.
    
- search_type: If using 'compound' modality, we can define the type of search to perform. Options are 'activity' (to search for compounds with specific bioactivity) or 'target' (to search for compounds targeting specific proteins).
   
- interaction_type: If using 'interaction' modality, we can define the type of interaction to retrieve. Options are 'protein-protein' or 'protein-ligand'.
   
- export_format: The format in which to export the retrieved data. Options include 'csv', 'json', 'xml', 'parquet'.

## Search examples

### Compounds with specific bioactivity ic50 between 20 and 30 nM that have an associated alphafold structure

In [2]:
data, meta = wf.run(
    mode="query_first",
    modality="compound",
    search_type="activity",
    query="ic50:20-30 AND databases:alphafold",
    export_format="csv",
    enrich=True,
)

2026-02-05 19:18:11 | INFO     | silkroute.core.workflow.main | Pipeline: fetching ChEMBL for query=ic50>20 AND ic50<30 search_type=activity
2026-02-05 19:18:12 | WARNING  | silkroute.core.workflow.main | Pipeline: large number of ChEMBL IDs (282); UniProt query may be too long
2026-02-05 19:18:12 | INFO     | silkroute.core.workflow.main | Searches will be divided into chunks of 100 IDs
2026-02-05 19:18:12 | INFO     | silkroute.core.workflow.main | Pipeline: fetching UniProt for query=(database:alphafolddb) AND (xref:chembl-CHEMBL2189139 OR xref:chembl-CHEMBL203 OR xref:chembl-CHEMBL325 OR xref:chembl-CHEMBL3524 OR xref:chembl-CHEMBL3192 OR xref:chembl-CHEMBL1829 OR xref:chembl-CHEMBL4145 OR xref:chembl-CHEMBL1937 OR xref:chembl-CHEMBL1865 OR xref:chembl-CHEMBL3145 OR xref:chembl-CHEMBL612545 OR xref:chembl-CHEMBL613740 OR xref:chembl-CHEMBL2997 OR xref:chembl-CHEMBL4078 OR xref:chembl-CHEMBL4354 OR xref:chembl-CHEMBL375 OR xref:chembl-CHEMBL5936 OR xref:chembl-CHEMBL2039 OR xref:che

Using workflow will automatically handle the chaining of requests and data retrieval based on the defined parameters.

The return format are separated in 2 parts:
1. Data: The main data retrieved from the queries.
2. Metadata: Additional information about the retrieval process, such as query parameters, timestamps, and any errors encountered.

Data is distributed in a dictionary, where each key represents a different source of data, for example:
- 'uniprot' for protein sequences from UniProt
- 'chembl' for compound structures from ChEMBL
- 'uniprot_enrichment' for enriched protein data from UniProt

Uniprot_enrichment will only be present if enrichment is set to True.
If so, the dictionary will contain every enrichment used in 'uniprot_enrichment' key separated by '{source}_{endpoint}'.

In [3]:
data.keys()

dict_keys(['chembl', 'uniprot', 'uniprot_enrichment'])

In [6]:
data["uniprot_enrichment"].keys()

dict_keys(['brenda_getTemperatureOptimum', 'brenda_getTemperatureStability', 'brenda_getTemperatureRange'])

In [4]:
data["uniprot"].head(5)

,accession,protein_name,ec,organism_name,gene_primary,organism_id,lineage,sequence,length,alphafold_ids,...,sabiork_ids,string_ids,references,active_sites,temperature,ph,domains,variants,interactions,keyword
0,O00591,Gamma-aminobutyric acid receptor subunit pi,None,Homo sapiens,[GABRP],9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",MNYSLHLAFVCLSLFTERMCIQGSQFNVEVGRSDKLSLPGFENLTA...,440,[O00591],...,[],[9606.ENSP00000430100],[{'title': 'A novel class of GABAA receptor su...,[],[],[],[],"[{'type': 'Natural variant', 'id': 'VAR_020323...",[],"[3D-structure, Cell membrane, Chloride, Chlori..."
1,O00748,Cocaine esterase,[3.1.1.84],Homo sapiens,[CES2],9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",MRLHRLRARLSAVACGLLLLLVRGQGQDSASPIRTTHTGQVLGSLV...,559,[O00748],...,[O00748],[9606.ENSP00000499140],[{'title': 'Molecular cloning and characteriza...,"[{'type': 'Active site', 'description': 'Acyl-...",[],[],"[{'type': 'Motif', 'description': 'Prevents se...","[{'type': 'Natural variant', 'id': 'VAR_018396...","[{'accesion_a': 'O00748', 'geneName_a': '', 'a...","[Alternative initiation, Alternative splicing,..."
2,O09028,Gamma-aminobutyric acid receptor subunit pi,None,Rattus norvegicus,[Gabrp],10116,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",MSYSLYLAFVCLNLLAQRMCIQGNQFNVEVSRSDKLSLPGFENLTA...,440,[O09028],...,[],[10116.ENSRNOP00000048081],[{'title': 'A novel class of GABAA receptor su...,[],[],[],[],[],[],"[Cell membrane, Chloride, Chloride channel, Di..."
3,O15379,Histone deacetylase 3,[3.5.1.98],Homo sapiens,[HDAC3],9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",MAKTVAYFYDPDVGNFHYGAGHPMKPHRLALTHSLVLHYGLYKKMI...,428,[O15379],...,[O15379],[9606.ENSP00000302967],[{'title': 'Differential display cloning of a ...,"[{'type': 'Active site', 'description': '', 'l...",[],[],"[{'type': 'Region', 'description': 'Histone de...","[{'type': 'Natural variant', 'id': 'VAR_033988...","[{'accesion_a': 'O15379', 'geneName_a': '', 'a...","[3D-structure, Alternative splicing, Biologica..."
4,O42275,Acetylcholinesterase,[3.1.1.7],Electrophorus electricus,[ache],8005,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",MKILDALLFPVIFIMFFIHLSIAQTDPELTIMTRLGQVQGTRLPVP...,633,[O42275],...,[O42275],[8005.ENSEEEP00000024330],[{'title': 'Cloning and expression of acetylch...,"[{'type': 'Active site', 'description': 'Acyl-...",[],[],[],[],[],"[Cell membrane, Disulfide bond, Glycoprotein, ..."


### Search for proteins with a specific temperature stability and retrieve their sequences

In [5]:
data, meta = wf.run(
    mode="query_first", modality="protein", query="temperature:98", export_format="csv", enrich=True
)

2026-02-05 19:48:12 | INFO     | silkroute.core.workflow.main | Pipeline: fetching UniProt for query=cc_bpcp_temp_dependence:98 fields=
2026-02-05 19:48:14 | INFO     | silkroute.core.workflow.main | Pipeline: performing CrossRef enrichment with fields=['brenda_getTemperatureOptimum', 'brenda_getTemperatureStability', 'brenda_getTemperatureRange']
2026-02-05 19:48:14 | INFO     | silkroute.core.utils.crossref_enrichment | Running crossref enrichment for fields: ['brenda_getTemperatureOptimum', 'brenda_getTemperatureStability', 'brenda_getTemperatureRange']
Processed crossref fields: {'brenda': [{'method': 'getTemperatureOptimum', 'option': None}, {'method': 'getTemperatureStability', 'option': None}, {'method': 'getTemperatureRange', 'option': None}]}
Crossref fields: ['brenda_getTemperatureOptimum', 'brenda_getTemperatureStability', 'brenda_getTemperatureRange']
2026-02-05 19:48:14 | INFO     | silkroute.interfaces.crossref_enricher | Checking availability for interface: brenda
2026-0

In [7]:
data["uniprot"].head(5)

,accession,protein_name,ec,organism_name,gene_primary,organism_id,lineage,sequence,length,alphafold_ids,...,sabiork_ids,string_ids,references,active_sites,temperature,ph,domains,variants,interactions,keyword
0,O08342,Endoglucanase A,[3.2.1.4],Paenibacillus barcinonensis,[celA],198119,"[Bacteria, Bacillati, Bacillota, Bacilli, Baci...",MTKTFKKFSIAGLALLFMATAAFAGWSTKASAADMRSLTAAQITAE...,400,[O08342],...,[],[],[{'title': 'Cloning of a new endoglucanase gen...,"[{'type': 'Active site', 'description': 'Proto...",[Optimum temperature is 40 degrees Celsius. Sh...,[Optimum pH is 4.0. Highly stable at acid pH. ...,[],[],[],"[Carbohydrate metabolism, Cellulose degradatio..."
1,P84142,Acylphosphatase,[3.6.1.7],Pyrococcus horikoshii (strain ATCC 700860 / DS...,[acyP],70601,"[Archaea, Methanobacteriati, Methanobacteriota...",MAIVRAHLKIYGRVQGVGFRWSMQREARKLGVNGWVRNLPDGSVEA...,91,[P84142],...,[P84142],[],[{'title': 'Complete sequence and gene organiz...,"[{'type': 'Active site', 'description': '', 'l...",[Optimum temperature is 98 degrees Celsius. Po...,[Optimum pH is 5.3 at 25 degrees Celsius.],"[{'type': 'Domain', 'description': 'Acylphosph...",[],[],"[3D-structure, Hydrolase, Reference proteome]"
2,P85124,Small basic protein 2,None,Anas platyrhynchos,None,8839,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",QVRKYCPKVGYCSSKCSKADVWSLSSDCKFYCCLPPGWK,39,[P85124],...,[],[],[{'title': 'Structural and physicochemical cha...,[],[Denaturation temperature (Td) is 98.3 degrees...,[],[],[],[],"[Direct protein sequencing, Disulfide bond, Py..."
3,Q58653,2-phospho-L-lactate transferase,[2.7.8.28],Methanocaldococcus jannaschii (strain ATCC 430...,[cofD],243232,"[Archaea, Methanobacteriati, Methanobacteriota...",MIFVITVLSGGTGTPKLLQGLKRVVNNEELAVIVNTGEDTWIGDLY...,311,[Q58653],...,[Q58653],[243232.MJ_1256],[{'title': 'Complete genome sequence of the me...,"[{'type': 'Binding site', 'description': '', '...",[Optimum temperature is about 37 degrees Celsi...,[],[],"[{'type': 'Mutagenesis', 'id': '', 'location':...",[],"[Magnesium, Reference proteome, Transferase]"
4,Q5S260,N-carbamoyl-D-amino acid hydrolase,[3.5.1.77],Ensifer adhaerens,None,106592,"[Bacteria, Pseudomonadati, Pseudomonadota, Alp...",MTRQMILAVGQQGPIARAETREQVVVRLLYMLTKAASRGANFIVFP...,304,[Q5S260],...,[],[],[{'title': 'Thermostable D-carbamoylase from S...,"[{'type': 'Active site', 'description': '', 'l...",[Optimum temperature is 60 degrees Celsius. Ve...,[Optimum pH is 7.0. Stable at pH 6.5-8.2.],"[{'type': 'Domain', 'description': 'CN hydrola...",[],[],"[Direct protein sequencing, Hydrolase]"


In [8]:
data["uniprot_enrichment"]["brenda_getTemperatureOptimum"]

,ec,organism,temperature_optimum,temperature_optimum_max
0,3.2.1.4,Aspergillus niger,-999,None
1,3.2.1.4,Rasamsonia emersonii,-999,None
2,3.2.1.4,Stegonsporium opalus,-999,None
3,3.2.1.4,Fibrobacter succinogenes,25,None
4,3.2.1.4,Marinobacter sp.,27,35
...,...,...,...,...
591,2.4.1.1,Sulfurisphaera tokodaii str. 7,75,None
592,2.4.1.1,Pyrococcus furiosus,80,None
593,2.4.1.1,Acetivibrio thermocellus,80,None
594,2.4.1.1,Thermus aquaticus,80,85


### Search ppi interactions for proteins reported in Prostate cancer (DI-02663)

In [9]:
data, meta = wf.run(
    mode="query_first",
    modality="interaction",
    export_format="csv",
    query="(cc_disease:DI-02663)",
    interaction_type="protein-protein",
)

2026-02-05 20:21:13 | INFO     | silkroute.core.workflow.main | Pipeline: fetching UniProt for query=( cc_disease:DI-02663 ) fields=
2026-02-05 20:21:15 | INFO     | silkroute.core.workflow.main | Pipeline: fetching additional interaction sources for protein-protein interactions
2026-02-05 20:21:15 | INFO     | silkroute.interfaces.crossref_enricher | Checking availability for interface: biogrid
2026-02-05 20:21:15 | INFO     | silkroute.interfaces.crossref_enricher | Checking required columns for biogrid:interactions...
2026-02-05 20:21:15 | INFO     | silkroute.interfaces.crossref_enricher | Building interface for biogrid...
2026-02-05 20:21:15 | INFO     | silkroute.interfaces.crossref_enricher | Prepared params for biogrid:interactions: {}
2026-02-05 20:21:28 | INFO     | silkroute.interfaces.crossref_enricher | Checking availability for interface: string
2026-02-05 20:21:28 | INFO     | silkroute.interfaces.crossref_enricher | Checking required columns for string:interaction_partn

In [10]:
data["uniprot"].head(5)

,accession,protein_name,ec,organism_name,gene_primary,organism_id,lineage,sequence,length,alphafold_ids,...,sabiork_ids,string_ids,references,active_sites,temperature,ph,domains,variants,interactions,keyword
0,O96017,Serine/threonine-protein kinase Chk2,[2.7.11.1],Homo sapiens,[CHEK2],9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",MSRESDVEAQQSHGSSACSQPHGSVTQSQGSSSQSQGISSSSTSTM...,543,[O96017],...,[],[9606.ENSP00000372023],[{'title': 'Linkage of ATM to cell cycle regul...,"[{'type': 'Active site', 'description': 'Proto...",[],[],"[{'type': 'Domain', 'description': 'FHA', 'loc...","[{'type': 'Natural variant', 'id': 'VAR_019101...","[{'accesion_a': 'O96017', 'geneName_a': '', 'a...","[3D-structure, Alternative splicing, Apoptosis..."
1,P21757,Macrophage scavenger receptor types I and II,None,Homo sapiens,[MSR1],9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",MEQWDHFHNQQEDTDSCSESVKFDARSMTALLPPNPKNSPSLQEKL...,451,[P21757],...,[],[9606.ENSP00000405453],[{'title': 'Human macrophage scavenger recepto...,[],[],[],"[{'type': 'Domain', 'description': 'Collagen-l...","[{'type': 'Natural variant', 'id': 'VAR_025190...","[{'accesion_a': 'P21757', 'geneName_a': '', 'a...","[3D-structure, Alternative splicing, Coiled co..."
2,P29323,Ephrin type-B receptor 2,[2.7.10.1],Homo sapiens,[EPHB2],9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",MALRRLGAALLLLPLLAAVEETLMDSTTATAELGWMVHPPSGWEEV...,1055,[P29323],...,[],[9606.ENSP00000383053],"[{'title': 'Overexpression of ERK, an EPH fami...","[{'type': 'Active site', 'description': 'Proto...",[],[],"[{'type': 'Domain', 'description': 'Eph LBD', ...","[{'type': 'Natural variant', 'id': 'VAR_032853...","[{'accesion_a': 'P29323', 'geneName_a': '', 'a...","[3D-structure, Alternative splicing, ATP-bindi..."
3,P50539,Max-interacting protein 1,None,Homo sapiens,[MXI1],9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",MERVKMINVQRLLEAAEFLERRERECEHGYASSFPSMPSPRLQHSK...,228,[P50539],...,[],[9606.ENSP00000331152],"[{'title': 'Mxi1, a protein that specifically ...",[],[],[],"[{'type': 'Domain', 'description': 'bHLH', 'lo...","[{'type': 'Natural variant', 'id': 'VAR_004499...","[{'accesion_a': 'P50539', 'geneName_a': '', 'a...","[Alternative splicing, Disease variant, DNA-bi..."
4,P60484,"Phosphatidylinositol 3,4,5-trisphosphate 3-pho...","[3.1.3.16, 3.1.3.48, 3.1.3.67]",Homo sapiens,[PTEN],9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",MTAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVY...,403,[P60484],...,[],[9606.ENSP00000361021],"[{'title': 'TEP1, encoded by a candidate tumor...","[{'type': 'Active site', 'description': 'Phosp...",[],[],"[{'type': 'Domain', 'description': 'Phosphatas...","[{'type': 'Natural variant', 'id': 'VAR_026248...","[{'accesion_a': 'P60484', 'geneName_a': '', 'a...","[3D-structure, Acetylation, Alternative initia..."


In [12]:
data["uniprot_enrichment"]["biogrid_interactions"]

,interaction_b,synonyms_a,synonyms_b,organism_a,organism_b
0,CHEK2,COCA1|FCC1|HNPCC|HNPCC1|LCFS2,CDS1|CHK2|HuCds1|LFS2|PP1425|RAD53|hCds1,9606,9606
1,BRCA1,CDS1|CHK2|HuCds1|LFS2|PP1425|RAD53|hCds1,BRCAI|BRCC1|BROVCA1|FANCS|IRIS|PNCA4|PPP1R53|P...,9606,9606
2,BRCA1,CDS1|CHK2|HuCds1|LFS2|PP1425|RAD53|hCds1,BRCAI|BRCC1|BROVCA1|FANCS|IRIS|PNCA4|PPP1R53|P...,9606,9606
3,CHEK2,NFBD1,CDS1|CHK2|HuCds1|LFS2|PP1425|RAD53|hCds1,9606,9606
4,CHEK2,NFBD1,CDS1|CHK2|HuCds1|LFS2|PP1425|RAD53|hCds1,9606,9606
...,...,...,...,...,...
2072,STRAP,BCD1|CBA1|COPEB|CPBP|GBF|PAC1|ST12|ZF9,MAWD|PT-WD|UNRIP,9606,9606
2073,KLF6,CEB1|CEBP1,BCD1|CBA1|COPEB|CPBP|GBF|PAC1|ST12|ZF9,9606,9606
2074,ACTA2,BCD1|CBA1|COPEB|CPBP|GBF|PAC1|ST12|ZF9,AAT6|ACTSA|MYMY5,9606,9606
2075,TANGO6,BCD1|CBA1|COPEB|CPBP|GBF|PAC1|ST12|ZF9,TMCO7,9606,9606


### Search pli interactions related to 'Human inmunodefificiency virus'

In [13]:
data, meta = wf.run(
    mode="query_first",
    modality="interaction",
    export_format="csv",
    query="Human immunodeficiency virus",
    interaction_type="protein-ligand",
)

2026-02-05 20:22:30 | INFO     | silkroute.core.workflow.main | Pipeline: fetching ChEMBL for query=Human immunodeficiency virus search_type=target
2026-02-05 20:22:30 | INFO     | silkroute.core.workflow.main | Pipeline: fetching UniProt for query=(xref:chembl-CHEMBL4523214 OR xref:chembl-CHEMBL613758 OR xref:chembl-CHEMBL378 OR xref:chembl-CHEMBL380 OR xref:chembl-CHEMBL612359 OR xref:chembl-CHEMBL613886 OR xref:chembl-CHEMBL2909 OR xref:chembl-CHEMBL3800 OR xref:chembl-CHEMBL243 OR xref:chembl-CHEMBL3471 OR xref:chembl-CHEMBL3852 OR xref:chembl-CHEMBL3463 OR xref:chembl-CHEMBL247 OR xref:chembl-CHEMBL3556 OR xref:chembl-CHEMBL5074 OR xref:chembl-CHEMBL613499 OR xref:chembl-CHEMBL613744 OR xref:chembl-CHEMBL613498 OR xref:chembl-CHEMBL613737 OR xref:chembl-CHEMBL4630879 OR xref:chembl-CHEMBL613551 OR xref:chembl-CHEMBL2362987 OR xref:chembl-CHEMBL3638323 OR xref:chembl-CHEMBL4296312 OR xref:chembl-CHEMBL4295909 OR xref:chembl-CHEMBL6066901 OR xref:chembl-CHEMBL248 OR xref:chembl-CHEM

In [15]:
data["uniprot"].head(5)

,accession,protein_name,ec,organism_name,gene_primary,organism_id,lineage,sequence,length,alphafold_ids,...,sabiork_ids,string_ids,references,active_sites,temperature,ph,domains,variants,interactions,keyword
0,D4A7K7,G-protein coupled receptor 183,None,Rattus norvegicus,[Gpr183],10116,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",MANNFTTPLAASHGNNCDLYAHHSTARILMPLHYSLVFIIGLVGNL...,357,[D4A7K7],...,[],[10116.ENSRNOP00000035137],[{'title': 'Genome sequence of the Brown Norwa...,"[{'type': 'Binding site', 'description': '', '...",[],[],"[{'type': 'Region', 'description': 'Interactio...",[],[],"[Adaptive immunity, Cell membrane, Disulfide b..."
1,P01911,"HLA class II histocompatibility antigen, DRB1 ...",None,Homo sapiens,[HLA-DRB1],9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",MVCLKLPGGSCMTALTVTLMVLSSPLALSGDTRPRFLWQPKRECHF...,266,[P01911],...,[],[9606.ENSP00000353099],[{'title': 'Mutations and selection in the gen...,"[{'type': 'Binding site', 'description': '', '...",[],[],"[{'type': 'Domain', 'description': 'Ig-like C1...","[{'type': 'Natural variant', 'id': 'VAR_082703...","[{'accesion_a': 'P01911', 'geneName_a': '', 'a...","[3D-structure, Adaptive immunity, Cell membran..."
2,P04439,"HLA class I histocompatibility antigen, A alph...",None,Homo sapiens,[HLA-A],9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",MAVMAPRTLLLLLSGALALTQTWAGSHSMRYFFTSVSRPGRGEPRF...,365,[P04439],...,[],[9606.ENSP00000379873],[{'title': 'The primary structure of HLA-A32 s...,"[{'type': 'Binding site', 'description': '', '...",[],[],"[{'type': 'Domain', 'description': 'Ig-like C1...","[{'type': 'Natural variant', 'id': 'VAR_082315...","[{'accesion_a': 'P04439', 'geneName_a': '', 'a...","[3D-structure, Adaptive immunity, Alternative ..."
3,P05412,Transcription factor Jun,None,Homo sapiens,[JUN],9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",MTAKMETTFYDDALNASFLPSESGPYGYSNPKILKQSMTLNLADPV...,331,[P05412],...,[],[9606.ENSP00000360266],[{'title': 'Structure and chromosomal localiza...,"[{'type': 'Site', 'description': 'Necessary fo...",[],[],"[{'type': 'Domain', 'description': 'bZIP', 'lo...","[{'type': 'Natural variant', 'id': 'VAR_012070...","[{'accesion_a': 'P05412', 'geneName_a': '', 'a...","[3D-structure, Acetylation, Activator, Direct ..."
4,P05627,Transcription factor Jun,None,Mus musculus,[Jun],10090,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",MTAKMETTFYDDALNASFLQSESGAYGYSNPKILKQSMTLNLADPV...,334,[P05627],...,[],[10090.ENSMUSP00000102711],[{'title': 'Transcriptional activation of c-ju...,"[{'type': 'Site', 'description': 'Necessary fo...",[],[],"[{'type': 'Domain', 'description': 'bZIP', 'lo...",[],"[{'accesion_a': 'P05627', 'geneName_a': '', 'a...","[Acetylation, Activator, DNA-binding, Isopepti..."


In [17]:
data["chembl"]

,cross_references,organism,pref_name,score,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Homo sapiens,Transcription factor HIVEP2,36.0,False,CHEMBL4523214,"{'accession': 'P31629', 'component_description...",SINGLE PROTEIN,9606
1,[],Human immunodeficiency virus,Human immunodeficiency virus,31.0,False,CHEMBL613758,[],ORGANISM,12721
2,[],Human immunodeficiency virus 1,Human immunodeficiency virus 1,28.0,False,CHEMBL378,[],ORGANISM,11676
3,[],Human immunodeficiency virus 2,Human immunodeficiency virus 2,28.0,False,CHEMBL380,[],ORGANISM,11709
4,[],Human immunodeficiency virus 3,Human immunodeficiency virus 3,28.0,False,CHEMBL612359,[],ORGANISM,35274
...,...,...,...,...,...,...,...,...,...
95,[],Mammarenavirus juninense,Junin virus,9.0,False,CHEMBL4523072,[],ORGANISM,2169991
96,[],Powassan virus,Powassan virus,9.0,False,CHEMBL4888473,[],ORGANISM,11083
97,[],Una virus,Una virus,9.0,False,CHEMBL4888479,[],ORGANISM,59304
98,[],Oropouche virus,Oropouche virus,9.0,False,CHEMBL5291624,[],ORGANISM,118655
